In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("USE CATALOG db_novacart")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_schema")

In [0]:
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.now().strftime("%Y-%m-%d")

print("Current gold_run_id : ", gold_run_id)
print("Run timestamp is : ", run_ts_str)

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS gold_schema.gold_ingestion_control(
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
          )
    USING DELTA
""")

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    
    if spark.catalog.tableExists(target_table):

        dt = DeltaTable.forName(spark, target_table)

        dt.alias("target")\
            .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()

    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name):
    
    ctrl = spark.table("gold_schema.gold_ingestion_control")\
        .filter(
            (col("layer") == "gold") &
            (col("entity_name") == entity_name) &
            (col("run_status") == "success")
        )\
        .orderBy(col("updated_at").desc())\
        .limit(1)

    rows = ctrl.collect()

    if not rows:
        return None
    
    else:
        return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_silver_ts, rows_merged):
    
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_silver_ts,
            int(rows_merged),
            "success",
            gold_run_id,
            datetime.now()    
        )],
        schema = """
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "gold_schema.gold_ingestion_control")

    dt.alias("t")\
        .merge(ctrl_df.alias("s"), f"t.layer = s.layer AND t.entity_name = s.entity_name")\
        .whenMatchedUpdate(
            set = {
                "last_processed_silver_run_id"  :   "s.last_processed_silver_run_id",
                "last_processed_silver_run_ts"  :   "s.last_processed_silver_run_ts",
                "rows_merged"                   :   "s.rows_merged",
                "run_status"                    :   "s.run_status",
                "gold_run_id"                   :   "s.gold_run_id",
                "updated_at"                    :   "s.updated_at"
            }
        )\
        .whenNotMatchedInsertAll()\
        .execute()

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print("Last processed silver run timestamp for gold : ", last_gold_ts)

silver_orders_current = spark.read.table("silver_schema.orders_transformed")
silver_products_current = spark.read.table("silver_schema.products_transformed")
silver_payments_current = spark.read.table("silver_schema.payments_transformed")

if last_gold_ts is None:

    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current

else:

    changed_orders = silver_orders_current.filter(col("updated_at") > last_gold_ts)
    changed_products = silver_products_current.filter(col("updated_at") > last_gold_ts)
    changed_payments = silver_payments_current.filter(col("updated_at") > last_gold_ts)

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print("Number of changed orders : ", changed_orders_count)
print("Number of changed products : ", changed_products_count)
print("Number of changed payments : ", changed_payments_count)

In [0]:
impacted_orders = changed_orders.select("order_id").distinct()
impacted_payments = changed_payments.select("order_id").distinct()
impacted_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"), col("p.product_id") == col("o.product_id"), "inner")
    .select(col("o.order_id"))
    .distinct()
)

impacted_order_ids = (
    impacted_orders
    .union(impacted_payments)
    .union(impacted_products)
    .distinct()
)

print("impacted-order_ids : ", impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))

In [0]:
impacted_orders_delta = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders_delta.alias("o")
    .join(silver_products_current.alias("p"), col("o.product_id") == col("p.product_id"), "inner")
    .join(silver_payments_current.alias("py"), col("o.order_id") == col("py.order_id"), "inner")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("p.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("p.price").alias("product_price"),
        col("o.order_status"),
        col("o.order_amount"),
        col("py.payment_id"),
        col("py.payment_status"),
        col("py.paid_amount"),
        col("o.order_date"),
        col("o.order_month"),
        col("o.order_year"),
        greatest(
            col("o.updated_at").cast("timestamp"),
            col("p.updated_at").cast("timestamp"),
            col("py.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
            when(
                col("order_amount") > 0,
                col("paid_amount") / col("order_amount")
            ).otherwise(lit(0.0))    
    )
    .withColumn(
        "payment_state",
        when(col("order_amount") == 0, "Invalid_order_amount")
        .when(col("payment_completion_ratio") == 0, "Unpaid")
        .when(col("payment_completion_ratio") == 1, "Paid")
        .when(col("payment_completion_ratio") < 1, "Partially_paid")
        .when(col("payment_completion_ratio") > 1, "Overpaid")
    )
    .withColumn("gold_updated_date", to_date(col("gold_update_ts")))
    .withColumn("gold_run_id", lit(gold_run_id))
)

print("gold_delta_rows : ", gold_delta.count())
display(gold_delta)

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "gold_schema.orders_information", "order_id")
else:
    print("No new records to insert in gold table")

In [0]:
if not spark.catalog.tableExists("gold_schema.orders_information_scd2"):
    spark.sql("""
              CREATE TABLE gold_schema.orders_information_scd2 
              USING DELTA AS 
              SELECT *,
                        cast(null as TIMESTAMP) as valid_from_ts,
                        cast(null as TIMESTAMP) as valid_to_ts,
                        true as is_current
                        from db_novacart.gold_schema.orders_information
                        where 1 = 0
            """)

if gold_delta.count() > 0:
    
    gold_delta.createOrReplaceTempView("gold_delta_view")

    spark.sql("""
                MERGE INTO gold_schema.orders_information_scd2 t
                USING gold_delta_view s
                ON t.order_id = s.order_id and t.is_current = true
                WHEN MATCHED AND (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price)
                )

                THEN UPDATE
                SET is_current = false,
                    valid_to_ts = s.gold_update_ts
            """)

    spark.sql("""
                INSERT INTO gold_schema.orders_information_scd2
                SELECT s.*,
                    s.gold_update_ts as valid_from_ts,
                    cast(null as TIMESTAMP) as valid_to_ts,
                    true as is_current
                FROM gold_delta_view s
                LEFT JOIN gold_schema.orders_information_scd2 t
                ON s.order_id = t.order_id and t.is_current = true
                WHERE t.order_id is null or (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price)
                )
            """)

In [0]:
%sql
select * from gold_schema.orders_information_scd2

In [0]:
if gold_delta.count() > 0:

    impacted_categories = (
        gold_delta.select("category")
        .filter(col("category").isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read.table("gold_schema.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            
            sum(
                when(col("order_amount") > 0, col("order_amount")).otherwise(lit(0.0))
            ).alias("Gross_Merchandise_Value"),
            
            sum(
                when(col("paid_amount") > 0, col("paid_amount")).otherwise(lit(0.0))
            ).alias("Total_Paid_Amount"),

            avg(
                col("payment_completion_ratio")
            ).alias("Average_Payment_Completion_Ratio"),

            (
                sum(when(col("payment_status") == "FAILED", 1).otherwise(lit(0))) / count("*")
            ).alias("Payment_Failure_Rate")
        )
    )

    upsert_to_gold(category_perf_delta, "gold_schema.category_performance", "category")

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS gold_schema.gold_snapshots_vol")

In [0]:
latest_orders_path = (
    "/Volumes/db_novacart/gold_schema/gold_snapshots_vol/gold_latest/order_information"
)

latest_category_path = (
    "/Volumes/db_novacart/gold_schema/gold_snapshots_vol/gold_latest/category_performance"
)

historical_orders_path = (
    f"/Volumes/db_novacart/gold_schema/gold_snapshots_vol/gold_historical/order_information/run_date={run_date_str}/run_ts={run_ts_str}"

)

historical_category_path = (
    f"/Volumes/db_novacart/gold_schema/gold_snapshots_vol/gold_historical/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"
)

spark.read.table("gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)
spark.read.table("gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print("Latest_order_path : ", latest_orders_path)
print("Latest_category_path : ", latest_category_path)
print("Historical_order_path : ", historical_orders_path)
print("Historical_category_path : ", historical_category_path)

In [0]:
latest_silver_ts = silver_orders_current.agg(max("bronze_ingested_at").alias("max_silver_ts")).collect()[0]["max_silver_ts"]

if latest_silver_ts is not None:
    latest_silver_run_id = (
            silver_orders_current
            .filter(col("bronze_ingested_at") == latest_silver_ts)
            .agg(max("silver_run_id").alias("max_silver_run_id"))
            .collect()[0]["max_silver_run_id"]
    )

else:
    latest_silver_run_id = None

upsert_gold_control("orders_information", latest_silver_run_id, latest_silver_ts, gold_delta.count())

display(spark.read.table("gold_schema.gold_ingestion_control"))